### Inferer whisper anglais sur le corpus dev du zazaki

In [9]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
from datasets import load_dataset, Audio
import evaluate
import torch
import os

In [16]:
def load_and_process_data (DATASETS_PATH, FOLDER, FILE_NAME):
    # Charger le dataset
    dataset = load_dataset('csv', data_files=os.path.join("..", "datasets", FOLDER, f"{FILE_NAME}.tsv"), sep="\t")
    PATH_TO_AUDIO = os.path.join(DATASETS_PATH, FOLDER, "clips/") 
    # Creer la colonne audio qui contient le chemin complet + le nom de fichier qui reside dans la colonne path
    dataset["train"] = dataset["train"].map( lambda x: {"audio": PATH_TO_AUDIO + x["path"]})
    """
    cast_columns: change le type d'une colonne
    Audio: permet de transformer un fichier audio en un tableau numpy avece un echantillonage de 16000 khz
    donc la colonne audio contiendra le tableau numpy qui sera apres transformer en spectogramme mel-log
    """
    dataset["train"] = dataset["train"].cast_column("audio", Audio(sampling_rate=16_000))
    return dataset

In [24]:
def whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE):
    
    # importer les metriques
    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")
    normalizer = BasicTextNormalizer()
    
    # pour execution sur gpu si dispo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    #le necessaire pour inferer, preparer le processeur (extracteur des caracteristiques acoustiques) et le decodeur etc..
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained(f"openai/whisper-{FAMILLE}")
    model = model.to(device)
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    # Recuperer le dataset
    ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)
    
    results = []
    compteur = 0
    
    # Iterer sur le dataset pour inferer
    for signal in ds["train"]:
        input_features = processor(
            signal['audio']['array'], 
            sampling_rate=signal['audio']['sampling_rate'], 
            return_tensors="pt"
        ).input_features
        
        predicted_ids = model.generate(
            input_features=input_features.to(device), 
            forced_decoder_ids=forced_decoder_ids
        )
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # Stocker les resultats
        results.append({
            "sentence": normalizer(signal['sentence']),  
            "predicted": normalizer(transcription)
        })
        
        compteur += 1
        
        # Afficher la progression
        if compteur % 50 == 0:
            print(f"Processing: {compteur} samples... whisper: {LANGUAGE} ... sur le corpus dev")
        
    
    references = [r["sentence"] for r in results]
    predictions = [r["predicted"] for r in results]
    
    # Calculer les metriques
    wer = 100 * wer_metric.compute(references=references, predictions=predictions)
    cer = 100 * cer_metric.compute(references=references, predictions=predictions)
    
    print(f"\n=== Results for zazaki ===")
    print(f"famille: {FAMILLE}")
    print(f"whisper:{LANGUAGE} ")
    print(f"Dataset: Common Voice")
    print(f"Split: dev ")
    print(f"Samples evaluated: {len(results)}")
    print(f"WER: {wer:.2f}%")
    print(f"CER: {cer:.2f}%")
    
    return wer, cer

##########################################################################
#####
#####         CORPUS DEV
#####     
############################################################################

### whisper small anglais

In [25]:
LANGUAGE = "english"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME)

Using device: cuda
Processing: 50 samples... whisper: english ... sur le corpus dev
Processing: 100 samples... whisper: english ... sur le corpus dev
Processing: 150 samples... whisper: english ... sur le corpus dev
Processing: 200 samples... whisper: english ... sur le corpus dev
Processing: 250 samples... whisper: english ... sur le corpus dev
Processing: 300 samples... whisper: english ... sur le corpus dev
Processing: 350 samples... whisper: english ... sur le corpus dev
Processing: 400 samples... whisper: english ... sur le corpus dev
Processing: 450 samples... whisper: english ... sur le corpus dev

=== Results for zazaki ===
Dataset: Common Voice
Split: dev 
Samples evaluated: 463
WER: 123.31%
CER: 64.31%


In [19]:
ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)

In [20]:
type(ds)

datasets.dataset_dict.DatasetDict

In [21]:
for d in ds:
    print(d)

train


### whisper base anglais

In [32]:
LANGUAGE = "english"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"
FAMILLE = "base"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: english ... sur le corpus dev
Processing: 100 samples... whisper: english ... sur le corpus dev
Processing: 150 samples... whisper: english ... sur le corpus dev
Processing: 200 samples... whisper: english ... sur le corpus dev
Processing: 250 samples... whisper: english ... sur le corpus dev
Processing: 300 samples... whisper: english ... sur le corpus dev
Processing: 350 samples... whisper: english ... sur le corpus dev
Processing: 400 samples... whisper: english ... sur le corpus dev
Processing: 450 samples... whisper: english ... sur le corpus dev

=== Results for zazaki ===
famille: base
whisper:english 
Dataset: Common Voice
Split: dev 
Samples evaluated: 463
WER: 206.24%
CER: 142.78%


### WHISPER TURC BASE

In [33]:
LANGUAGE = "turkish"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"
FAMILLE = "base"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: turkish ... sur le corpus dev
Processing: 100 samples... whisper: turkish ... sur le corpus dev
Processing: 150 samples... whisper: turkish ... sur le corpus dev
Processing: 200 samples... whisper: turkish ... sur le corpus dev
Processing: 250 samples... whisper: turkish ... sur le corpus dev
Processing: 300 samples... whisper: turkish ... sur le corpus dev
Processing: 350 samples... whisper: turkish ... sur le corpus dev
Processing: 400 samples... whisper: turkish ... sur le corpus dev
Processing: 450 samples... whisper: turkish ... sur le corpus dev

=== Results for zazaki ===
famille: base
whisper:turkish 
Dataset: Common Voice
Split: dev 
Samples evaluated: 463
WER: 107.78%
CER: 52.05%


### Whisper turc small 

In [36]:
LANGUAGE = "turkish"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"
FAMILLE = "small"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: turkish ... sur le corpus dev
Processing: 100 samples... whisper: turkish ... sur le corpus dev
Processing: 150 samples... whisper: turkish ... sur le corpus dev
Processing: 200 samples... whisper: turkish ... sur le corpus dev
Processing: 250 samples... whisper: turkish ... sur le corpus dev
Processing: 300 samples... whisper: turkish ... sur le corpus dev
Processing: 350 samples... whisper: turkish ... sur le corpus dev
Processing: 400 samples... whisper: turkish ... sur le corpus dev
Processing: 450 samples... whisper: turkish ... sur le corpus dev

=== Results for zazaki ===
famille: small
whisper:turkish 
Dataset: Common Voice
Split: dev 
Samples evaluated: 463
WER: 123.40%
CER: 54.45%


### WHISPER FRENCH SMALL

In [37]:
LANGUAGE = "french"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"
FAMILLE = "small"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: french ... sur le corpus dev
Processing: 100 samples... whisper: french ... sur le corpus dev
Processing: 150 samples... whisper: french ... sur le corpus dev
Processing: 200 samples... whisper: french ... sur le corpus dev
Processing: 250 samples... whisper: french ... sur le corpus dev
Processing: 300 samples... whisper: french ... sur le corpus dev
Processing: 350 samples... whisper: french ... sur le corpus dev
Processing: 400 samples... whisper: french ... sur le corpus dev
Processing: 450 samples... whisper: french ... sur le corpus dev

=== Results for zazaki ===
famille: small
whisper:french 
Dataset: Common Voice
Split: dev 
Samples evaluated: 463
WER: 221.29%
CER: 164.38%


### Whisper french BASE

In [38]:
LANGUAGE = "french"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "dev"
FAMILLE = "base"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: french ... sur le corpus dev
Processing: 100 samples... whisper: french ... sur le corpus dev
Processing: 150 samples... whisper: french ... sur le corpus dev
Processing: 200 samples... whisper: french ... sur le corpus dev
Processing: 250 samples... whisper: french ... sur le corpus dev
Processing: 300 samples... whisper: french ... sur le corpus dev
Processing: 350 samples... whisper: french ... sur le corpus dev
Processing: 400 samples... whisper: french ... sur le corpus dev
Processing: 450 samples... whisper: french ... sur le corpus dev

=== Results for zazaki ===
famille: base
whisper:french 
Dataset: Common Voice
Split: dev 
Samples evaluated: 463
WER: 637.89%
CER: 516.25%


##########################################################################
#####
#####         CORPUS TEST
#####     
############################################################################

In [47]:
def whisper_inference_test(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE):
    
    # importer les metriques
    wer_metric = evaluate.load("wer")
    cer_metric = evaluate.load("cer")
    normalizer = BasicTextNormalizer()
    
    # pour execution sur gpu si dispo
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    #le necessaire pour inferer, preparer le processeur (extracteur des caracteristiques acoustiques) et le decodeur etc..
    processor = WhisperProcessor.from_pretrained("openai/whisper-small")
    model = WhisperForConditionalGeneration.from_pretrained(f"openai/whisper-{FAMILLE}")
    model = model.to(device)
    forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

    # Recuperer le dataset
    ds = load_and_process_data(DATASETS_PATH, FOLDER, FILE_NAME)
    
    results = []
    compteur = 0
    
    # Iterer sur le dataset pour inferer
    for signal in ds["train"]:
        input_features = processor(
            signal['audio']['array'], 
            sampling_rate=signal['audio']['sampling_rate'], 
            return_tensors="pt"
        ).input_features
        
        predicted_ids = model.generate(
            input_features=input_features.to(device), 
            forced_decoder_ids=forced_decoder_ids
        )
        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        # Stocker les resultats
        results.append({
            "sentence": normalizer(signal['sentence']),  
            "predicted": normalizer(transcription)
        })
        
        compteur += 1
        
        # Afficher la progression
        if compteur % 50 == 0:
            print(f"Processing: {compteur} samples... whisper: {LANGUAGE} ... sur le corpus test")
        
    
    references = [r["sentence"] for r in results]
    predictions = [r["predicted"] for r in results]
    
    # Calculer les metriques
    wer = 100 * wer_metric.compute(references=references, predictions=predictions)
    cer = 100 * cer_metric.compute(references=references, predictions=predictions)
    
    print(f"\n=== Results for zazaki ===")
    print(f"famille: {FAMILLE}")
    print(f"whisper:{LANGUAGE} ")
    print(f"Dataset: Common Voice")
    print(f"Split: test ")
    print(f"Samples evaluated: {len(results)}")
    print(f"WER: {wer:.2f}%")
    print(f"CER: {cer:.2f}%")
    
    return wer, cer

### WHISPER ENGLISH BASE

In [40]:
LANGUAGE = "english"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"
FAMILLE = "base"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda


Generating train split: 392 examples [00:00, 12689.31 examples/s]
Map: 100%|██████████| 392/392 [00:00<00:00, 5591.19 examples/s]


Processing: 50 samples... whisper: english ... sur le corpus dev
Processing: 100 samples... whisper: english ... sur le corpus dev
Processing: 150 samples... whisper: english ... sur le corpus dev
Processing: 200 samples... whisper: english ... sur le corpus dev
Processing: 250 samples... whisper: english ... sur le corpus dev
Processing: 300 samples... whisper: english ... sur le corpus dev
Processing: 350 samples... whisper: english ... sur le corpus dev

=== Results for zazaki ===
famille: base
whisper:english 
Dataset: Common Voice
Split: dev 
Samples evaluated: 392
WER: 172.65%
CER: 109.42%


### WHISPER ENGLISH SMALL

In [41]:
LANGUAGE = "english"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"
FAMILLE = "small"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: english ... sur le corpus dev
Processing: 100 samples... whisper: english ... sur le corpus dev
Processing: 150 samples... whisper: english ... sur le corpus dev
Processing: 200 samples... whisper: english ... sur le corpus dev
Processing: 250 samples... whisper: english ... sur le corpus dev
Processing: 300 samples... whisper: english ... sur le corpus dev
Processing: 350 samples... whisper: english ... sur le corpus dev

=== Results for zazaki ===
famille: small
whisper:english 
Dataset: Common Voice
Split: dev 
Samples evaluated: 392
WER: 134.73%
CER: 78.05%


### WHISPER TURC SMALL

In [48]:
LANGUAGE = "turkish"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"
FAMILLE = "small"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference_test(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: turkish ... sur le corpus test
Processing: 100 samples... whisper: turkish ... sur le corpus test
Processing: 150 samples... whisper: turkish ... sur le corpus test
Processing: 200 samples... whisper: turkish ... sur le corpus test
Processing: 250 samples... whisper: turkish ... sur le corpus test
Processing: 300 samples... whisper: turkish ... sur le corpus test
Processing: 350 samples... whisper: turkish ... sur le corpus test

=== Results for zazaki ===
famille: small
whisper:turkish 
Dataset: Common Voice
Split: test 
Samples evaluated: 392
WER: 130.93%
CER: 86.73%


### whisper turc base

In [49]:
LANGUAGE = "turkish"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"
FAMILLE = "base"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference_test(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: turkish ... sur le corpus test
Processing: 100 samples... whisper: turkish ... sur le corpus test
Processing: 150 samples... whisper: turkish ... sur le corpus test
Processing: 200 samples... whisper: turkish ... sur le corpus test
Processing: 250 samples... whisper: turkish ... sur le corpus test
Processing: 300 samples... whisper: turkish ... sur le corpus test
Processing: 350 samples... whisper: turkish ... sur le corpus test

=== Results for zazaki ===
famille: base
whisper:turkish 
Dataset: Common Voice
Split: test 
Samples evaluated: 392
WER: 111.67%
CER: 49.94%


### whisper french small

In [50]:
LANGUAGE = "french"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"
FAMILLE = "small"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference_test(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: french ... sur le corpus test
Processing: 100 samples... whisper: french ... sur le corpus test
Processing: 150 samples... whisper: french ... sur le corpus test
Processing: 200 samples... whisper: french ... sur le corpus test
Processing: 250 samples... whisper: french ... sur le corpus test
Processing: 300 samples... whisper: french ... sur le corpus test
Processing: 350 samples... whisper: french ... sur le corpus test

=== Results for zazaki ===
famille: small
whisper:french 
Dataset: Common Voice
Split: test 
Samples evaluated: 392
WER: 196.59%
CER: 145.55%


In [52]:
LANGUAGE = "french"
TASK = "transcribe"
FOLDER = "zazaki/"
FILE_NAME = "test"
FAMILLE = "base"
DATASETS_PATH = "/info/raid-etu/m1/s2506992/projet-m1-asr/datasets/"
wer, cer = whisper_inference_test(DATASETS_PATH, LANGUAGE, TASK, FOLDER, FILE_NAME, FAMILLE)

Using device: cuda
Processing: 50 samples... whisper: french ... sur le corpus test
Processing: 100 samples... whisper: french ... sur le corpus test
Processing: 150 samples... whisper: french ... sur le corpus test
Processing: 200 samples... whisper: french ... sur le corpus test
Processing: 250 samples... whisper: french ... sur le corpus test
Processing: 300 samples... whisper: french ... sur le corpus test
Processing: 350 samples... whisper: french ... sur le corpus test

=== Results for zazaki ===
famille: base
whisper:french 
Dataset: Common Voice
Split: test 
Samples evaluated: 392
WER: 569.40%
CER: 419.59%
